In [2]:
import pandas as pd
import geopandas as gpd
import numpy as np
from scipy.spatial import cKDTree
import re
import os
import time
import requests
from dotenv import load_dotenv

Data Validation Utility

In [3]:
def validate_geodataframe(gdf, name="GeoDataFrame"):
    """
    Validate a GeoDataFrame before analysis.
    Checks for null geometries, invalid geometries, and CRS information.
    Optionally auto-fixes invalid geometries using buffer(0).
    
    Parameters:
    -----------
    gdf : geopandas.GeoDataFrame
        The GeoDataFrame to validate
    name : str
        A descriptive name for the dataset (used in print output)
    
    Returns:
    --------
    geopandas.GeoDataFrame
        The validated (and potentially fixed) GeoDataFrame
    """
    print(f"Validating '{name}'...")
    print(f"  - Total records: {len(gdf)}")
    print(f"  - CRS: {gdf.crs}")
    
    null_count = gdf.geometry.isnull().sum()
    invalid_count = (~gdf.geometry.is_valid).sum()
    
    print(f"  - Null geometries: {null_count}")
    print(f"  - Invalid geometries: {invalid_count}")
    
    if null_count > 0:
        print(f"  WARNING: Contains {null_count} null geometries - these rows may cause issues in spatial operations")
    
    if invalid_count > 0:
        print(f"  WARNING: Contains {invalid_count} invalid geometries - attempting to fix with buffer(0)...")
        gdf = gdf.copy()
        gdf['geometry'] = gdf.geometry.buffer(0)
        new_invalid_count = (~gdf.geometry.is_valid).sum()
        if new_invalid_count == 0:
            print(f"  SUCCESS: All geometries are now valid")
        else:
            print(f"  PARTIAL FIX: {new_invalid_count} geometries still invalid")
    
    print(f"  Validation complete.\n")
    return gdf

### 1. HDB resale price with coordinates

1.1 HDB resale price 2025

In [4]:
resale_price_df =  pd.read_csv("../datasets/Resale flat prices based on registration date from Jan-2017 onwards.csv")

In [5]:
def clean_street_name(street):
    if pd.isna(street): return street
    street = street.upper().strip()
    
    # Define common Singapore address abbreviations
    replacements = {
        r'\bBT\b': 'BUKIT',
        r'\bRD\b': 'ROAD',
        r'\bAVE\b': 'AVENUE',
        r'\bDR\b': 'DRIVE',
        r'\bCL\b': 'CLOSE',
        r'\bCTRL\b': 'CENTRAL',
        r'\bCPD\b': 'COMPOUND',
        r'\bCRES\b': 'CRESCENT',
        r'\bGRV\b': 'GROVE',
        r'\bHWY\b': 'HIGHWAY',
        r'\bJLN\b': 'JALAN',
        r'\bKG\b': 'KAMPONG',
        r'\bLN\b': 'LANE', 
        r'\bMT\b': 'MOUNT',
        r'\bPL\b': 'PLACE',
        r'\bRS\b': 'RISE',
        r'\bTER\b': 'TERRACE',
        r'\bTG\b': 'TANJONG',
        r'\bUPP\b': 'UPPER',
        r'\bNTH\b': 'NORTH',
        r'\bSTH\b': 'SOUTH',
        r'\bPK\b': 'PARK',
        r'\bMKT\b': 'MARKET',
        r"\bC'WEALTH\b": 'COMMONWEALTH',
        r"\bLOR\b": 'LORONG',
        r'\bHTS\b': 'HEIGHTS',
        r'\bGDNS\b': 'GARDENS',
        r'\bST\b(?!\.)': 'STREET'
    }
    
    for short, full in replacements.items():
        street = re.sub(short, full, street)
    
    # Remove extra spaces caused by regex
    return " ".join(street.split())

In [6]:
# filter only 2019-2025 resale data 
resale_price_df["month"]= pd.to_datetime(resale_price_df["month"])
resale_df = resale_price_df[(resale_price_df["month"].dt.year >= 2019) & (resale_price_df["month"].dt.year <= 2025)].copy()
resale_df['street_name'] = resale_df['street_name'].apply(clean_street_name)

# create year and month columns 
resale_df['year'] = resale_df['month'].dt.year
resale_df['month'] = resale_df['month'].dt.month
resale_df.head()

,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price,year
42070,1,ANG MO KIO,3 ROOM,225,ANG MO KIO AVENUE 1,01 TO 03,67.0,New Generation,1978,58 years,230000.0,2019
42071,1,ANG MO KIO,3 ROOM,174,ANG MO KIO AVENUE 4,01 TO 03,60.0,Improved,1986,66 years,235000.0,2019
42072,1,ANG MO KIO,3 ROOM,440,ANG MO KIO AVENUE 10,04 TO 06,67.0,New Generation,1979,59 years,238000.0,2019
42073,1,ANG MO KIO,3 ROOM,174,ANG MO KIO AVENUE 4,10 TO 12,61.0,Improved,1986,66 years 01 month,240000.0,2019
42074,1,ANG MO KIO,3 ROOM,637,ANG MO KIO AVENUE 6,01 TO 03,68.0,New Generation,1980,60 years 08 months,240000.0,2019


1.2 Existing HDBs

In [7]:
# Instead of scraping from OneMap, we can use the HDB Existing Building dataset which contains the coordinates for all HDB blocks. 
# We will match on block and street name to get the coordinates for each resale transaction.
hdb_existing = gpd.read_file("../datasets/HDBExistingBuilding.geojson")
road_namecode = pd.read_excel("../datasets/road_name_code.xlsx", skiprows= 10)

In [8]:
road_namecode = road_namecode[["Domain Value", "Description"]].copy()

# match on "ST_COD" = "Domain Value" to get road description
hdb_existing = hdb_existing.merge(
    road_namecode,
    left_on= "ST_COD", 
    right_on= "Domain Value", 
    how= "left"
)
# ensure all ST_COD has a match
unmatched_rows = hdb_existing[hdb_existing['Domain Value'].isna()]
if unmatched_rows.empty:
    print("All 'ST_COD' values found a match in the road name table.")
else:
    print(f"{len(unmatched_rows)} rows failed to match.")
    print("Missing ST_COD values:")
    print(unmatched_rows['ST_COD'].unique())
    
# rename and select columns
hdb_existing = hdb_existing.rename(columns={"Description": "STREET_NAME", "geometry": "GEOMETRY"})
hdb_existing = hdb_existing[["BLK_NO", "STREET_NAME", "GEOMETRY", "POSTAL_COD"]]
hdb_existing.head()

All 'ST_COD' values found a match in the road name table.


,BLK_NO,STREET_NAME,GEOMETRY,POSTAL_COD
0,208B,CLEMENTI AVENUE 6,"POLYGON ((103.76235 1.32274, 103.76232 1.32274...",122208
1,238,BUKIT BATOK EAST AVENUE 5,"POLYGON ((103.75446 1.3501, 103.75448 1.35011,...",650238
2,88,CIRCUIT ROAD,"POLYGON ((103.88589 1.32371, 103.88579 1.3237,...",370088
3,44A,MARINE CRESCENT,"POLYGON ((103.91325 1.30501, 103.91325 1.30503...",441044
4,671B,JURONG WEST STREET 65,"POLYGON ((103.70208 1.34397, 103.70208 1.34398...",642671


1.3 Merge existing HDBs geometry with resale prices

In [9]:
# standardize to string types
resale_df['block'] = resale_df['block'].astype(str).str.strip()
hdb_existing['BLK_NO'] = hdb_existing['BLK_NO'].astype(str).str.strip()
# match on block and street name to get coordinates
resale_with_geom_df = resale_df.merge(
    hdb_existing, 
    left_on=['block', 'street_name'], 
    right_on=['BLK_NO', 'STREET_NAME'], 
    how='left'
)
# rename and select columns
resale_with_geom_df = resale_with_geom_df.rename(columns={'GEOMETRY': 'geometry', 'POSTAL_COD': 'postal_code'})
selected_columns = [
    'month', 
    'year',
    'town', 
    'flat_type', 
    'storey_range', 
    'floor_area_sqm', 
    'flat_model', 
    'remaining_lease', 
    'resale_price', 
    'geometry',
    'postal_code'
]
resale_with_geom_df = resale_with_geom_df[selected_columns]

In [10]:
# convert dataframe to geodataframe
resale_gdf = gpd.GeoDataFrame(resale_with_geom_df, geometry='geometry')
# reproject from geographic CRS (WGS84) to a projected CRS using EPSG:3414 (SVY21)
resale_gdf = resale_gdf.to_crs(3414)
# convert hdb polygons to points (centroids)
resale_gdf['geometry'] = resale_gdf['geometry'].centroid
print(resale_gdf.head())

   month  year        town flat_type storey_range  floor_area_sqm  \
0      1  2019  ANG MO KIO    3 ROOM     01 TO 03            67.0   
1      1  2019  ANG MO KIO    3 ROOM     01 TO 03            60.0   
2      1  2019  ANG MO KIO    3 ROOM     04 TO 06            67.0   
3      1  2019  ANG MO KIO    3 ROOM     10 TO 12            61.0   
4      1  2019  ANG MO KIO    3 ROOM     01 TO 03            68.0   

       flat_model     remaining_lease  resale_price  \
0  New Generation            58 years      230000.0   
1        Improved            66 years      235000.0   
2  New Generation            59 years      238000.0   
3        Improved   66 years 01 month      240000.0   
4  New Generation  60 years 08 months      240000.0   

                      geometry postal_code  
0   POINT (28537.68 38825.233)      560225  
1  POINT (28487.136 39687.765)      560174  
2    POINT (30336.26 38718.18)      560440  
3  POINT (28487.136 39687.765)      560174  
4  POINT (29003.603 40255.313

### 2. Convert to real prices

For a transaction in quarter $q$:

$$
\text{resale\_price\_2025} = \text{resale\_price} \times \frac{\text{index}_{2025\text{-target}}}{\text{index}_q}
$$

In [11]:
# Load resale price index
price_index_df = pd.read_csv("../datasets/HDBResalePriceIndex1Q2009100Quarterly.csv")
price_index_df["quarter"] = price_index_df["quarter"].astype(str)
price_index_df["index"] = pd.to_numeric(price_index_df["index"], errors="coerce")

resale_2025_gdf = resale_gdf.copy()

# get 2025 average price index 
index_2025 = price_index_df[price_index_df["quarter"].str.startswith("2025-")]["index"]
avg_index_2025 = index_2025.mean()

# build transaction quarters in hdb resale data
resale_2025_gdf['transaction_quarter'] = resale_2025_gdf.apply(lambda row: f"{row['year']}-Q{((row['month']-1)//3)+1}", axis=1)
resale_2025_gdf.head()

# merge price index values into resale_df
resale_df = resale_2025_gdf.merge(
    price_index_df.rename(columns={"quarter": "transaction_quarter", "index": "px_index"}),
    on="transaction_quarter",
    how="left"
)

# normalize prices to 2025 dollars using the price index
resale_2025_gdf['resale_price_2025'] = resale_df["resale_price"] * (avg_index_2025 / resale_df["px_index"])
resale_2025_gdf["resale_price_2025"] = resale_2025_gdf["resale_price_2025"].round(2)

# drop columns and rename 
resale_2025_gdf = (resale_2025_gdf
    .drop(columns=['resale_price', 'transaction_quarter'], errors='ignore')
    .rename(columns={'resale_price_2025': 'resale_price'})
)

resale_2025_gdf.head()

,month,year,town,flat_type,storey_range,floor_area_sqm,flat_model,remaining_lease,geometry,postal_code,resale_price
0,1,2019,ANG MO KIO,3 ROOM,01 TO 03,67.0,New Generation,58 years,POINT (28537.68 38825.233),560225,356061.07
1,1,2019,ANG MO KIO,3 ROOM,01 TO 03,60.0,Improved,66 years,POINT (28487.136 39687.765),560174,363801.53
2,1,2019,ANG MO KIO,3 ROOM,04 TO 06,67.0,New Generation,59 years,POINT (30336.26 38718.18),560440,368445.80
3,1,2019,ANG MO KIO,3 ROOM,10 TO 12,61.0,Improved,66 years 01 month,POINT (28487.136 39687.765),560174,371541.98
4,1,2019,ANG MO KIO,3 ROOM,01 TO 03,68.0,New Generation,60 years 08 months,POINT (29003.603 40255.313),560637,371541.98


### 3. Get school coordinates via OneMap API

In [12]:
## Get centroid coordinates for schools using OneMap API
df = pd.read_csv("../school_scoring/schools_tiered.csv")

# output columns
for col in ["latitude", "longitude", "searchval", "formatted_address", "postal_found", "geocode_status", "matched_query"]:
    if col not in df.columns:
        df[col] = None

# import request
base_url = "https://www.onemap.gov.sg/api/common/elastic/search"

count = 0

for i in range(len(df)):
    if pd.notna(df.loc[i, "latitude"]) and pd.notna(df.loc[i, "longitude"]):
        print(f"Row {i}: already geocoded")
        continue

    school_name = str(df.loc[i, "school_name"]).strip() if pd.notna(df.loc[i, "school_name"]) else ""
    address = str(df.loc[i, "address"]).strip() if pd.notna(df.loc[i, "address"]) else ""
    postal_code = str(df.loc[i, "postal_code"]).strip() if pd.notna(df.loc[i, "postal_code"]) else ""

    queries = []

    if postal_code and postal_code.lower() != "nan":
        queries.append(("postal_code", postal_code))

    if address and address.lower() != "nan":
        queries.append(("address", f"{address} Singapore"))

    if school_name and school_name.lower() != "nan":
        queries.append(("school_name", school_name))

    result = None
    matched_query_type = None

    try:
        for query_type, q in queries:
            params = {
                "searchVal": q,
                "returnGeom": "Y",
                "getAddrDetails": "Y",
                "pageNum": 1
            }

            resp = requests.get(base_url, params=params, timeout=15)
            time.sleep(0.5)
            resp.raise_for_status()
            data = resp.json()

            if data.get("results") and len(data["results"]) > 0:
                result = data["results"][0]
                matched_query_type = query_type
                break

        if result is not None:
            df.loc[i, "longitude"] = result.get("LONGITUDE")
            df.loc[i, "latitude"] = result.get("LATITUDE")
            df.loc[i, "searchval"] = result.get("SEARCHVAL")
            df.loc[i, "formatted_address"] = result.get("ADDRESS")
            df.loc[i, "postal_found"] = result.get("POSTAL")
            df.loc[i, "geocode_status"] = "success"
            df.loc[i, "matched_query"] = matched_query_type

            print(
                f"{school_name}: success via {matched_query_type} | "
                f"lat={result.get('LATITUDE')}, lon={result.get('LONGITUDE')}"
            )
        else:
            df.loc[i, "geocode_status"] = "not_found"
            df.loc[i, "matched_query"] = None
            print(f"{school_name}: not found")

    except Exception as e:
        df.loc[i, "geocode_status"] = f"error: {str(e)}"
        df.loc[i, "matched_query"] = None
        print(f"{school_name}: error -> {e}")

    count += 1

    if count % 50 == 0:
        df.to_csv("schools_geocoded_partial.csv", index=False)
        print(f"Saved partial progress at row {i}. Sleeping for 2 seconds...")
        time.sleep(2)

# df.to_csv("schools_geocoded.csv", index=False)
schools_gdf = df
print("Done.")

NANYANG PRIMARY SCHOOL: success via postal_code | lat=1.32107094241153, lon=103.807681852799
CATHOLIC HIGH SCHOOL: success via postal_code | lat=1.35438878373608, lon=103.844210670766
CHIJ ST. NICHOLAS GIRLS' SCHOOL: success via postal_code | lat=1.37347687878916, lon=103.834253269436
TAO NAN SCHOOL: success via postal_code | lat=1.30528528687321, lon=103.911553152326
NAN HUA PRIMARY SCHOOL: success via postal_code | lat=1.31919648268012, lon=103.761179518517
AI TONG SCHOOL: success via postal_code | lat=1.36058343005814, lon=103.83302033725
ANGLO-CHINESE SCHOOL (PRIMARY): success via postal_code | lat=1.31837054523521, lon=103.835609732354
MARIS STELLA HIGH SCHOOL: success via postal_code | lat=1.34148765509926, lon=103.877853946625
ROSYTH SCHOOL: success via postal_code | lat=1.3728583654058, lon=103.87477170399
HOLY INNOCENTS' PRIMARY SCHOOL: success via postal_code | lat=1.36693830877349, lon=103.894114899795
PEI HWA PRESBYTERIAN PRIMARY SCHOOL: success via postal_code | lat=1.3380

In [13]:
# verify schools have valid geocode
print(schools_gdf[schools_gdf["geocode_status"] != "success"])

# reproject from geographic CRS (WGS84) to a projected CRS using EPSG:3414 (SVY21)
schools_gdf = gpd.GeoDataFrame(
    schools_gdf,
    geometry=gpd.points_from_xy(schools_gdf["longitude"], schools_gdf["latitude"]),
    crs="EPSG:4326"
)
schools_gdf = schools_gdf.to_crs(epsg=3414)

Empty DataFrame
Columns: [school_name, address, postal_code, nature_code, sap_ind, autonomous_ind, gifted_ind, P1_demand, P2A_demand, P2B_demand, P2C_demand, P2CS_demand, subject_count, distprog_count, cca_clubs, cca_others, cca_sports, cca_uniformed, cca_arts, affiliation_count, top_psle_score, school_score_raw, school_score, school_tier, latitude, longitude, searchval, formatted_address, postal_found, geocode_status, matched_query]
Index: []

[0 rows x 31 columns]


### 3. Get feature variables of schools

3.1 
- num_schools_1km
- num_schools_2km

In [14]:
# get unique hdb locations
unique_hdb = resale_2025_gdf.drop_duplicates(subset=["postal_code"]).copy()
key_col = "postal_code"

# prepare unique hdb and school coordinates for distance calculation
school_coords = np.array([(geom.x, geom.y) for geom in schools_gdf.geometry])
hdb_coords = np.array([(geom.x, geom.y) for geom in unique_hdb.geometry])

school_names = schools_gdf["school_name"].tolist()

# build cKDTree for efficient spatial queries
tree = cKDTree(school_coords)

# get schools within 1km and 2km
indices_1km = tree.query_ball_point(hdb_coords, r=1000)
indices_2km = tree.query_ball_point(hdb_coords, r=2000)

# convert index arrays into school name lists
unique_hdb["schools_within_1km"] = [
    [school_names[i] for i in idx_list] for idx_list in indices_1km]

unique_hdb["schools_within_2km"] = [
    [school_names[i] for i in idx_list] for idx_list in indices_2km]

# count columns
unique_hdb["num_schools_1km"] = unique_hdb["schools_within_1km"].apply(len)
unique_hdb["num_schools_2km"] = unique_hdb["schools_within_2km"].apply(len)

3.2 
- num_tier1_schools_1km
- num_tier1_schools_2km

In [15]:
tier1 = schools_gdf[schools_gdf["school_tier"] == "Tier 1"].copy()
tier1_coords = np.array([(geom.x, geom.y) for geom in tier1.geometry])
tier1_names = tier1["school_name"].tolist()

tier1_tree = cKDTree(tier1_coords)

tier1_idx_1km = tier1_tree.query_ball_point(hdb_coords, r=1000)
tier1_idx_2km = tier1_tree.query_ball_point(hdb_coords, r=2000)

unique_hdb["tier1_schools_within_1km"] = [
    [tier1_names[i] for i in idx_list] for idx_list in tier1_idx_1km]

unique_hdb["tier1_schools_within_2km"] = [
    [tier1_names[i] for i in idx_list] for idx_list in tier1_idx_2km]

unique_hdb["num_tier1_schools_1km"] = unique_hdb["tier1_schools_within_1km"].apply(len)
unique_hdb["num_tier1_schools_2km"] = unique_hdb["tier1_schools_within_2km"].apply(len)

3.3
- num_tier2_schools_1km
- num_tier2_schools_2km

In [16]:
tier2 = schools_gdf[schools_gdf["school_tier"] == "Tier 2"].copy()
tier2_coords = np.array([(geom.x, geom.y) for geom in tier2.geometry])
tier2_names = tier2["school_name"].tolist()

tier2_tree = cKDTree(tier2_coords)

tier2_idx_1km = tier2_tree.query_ball_point(hdb_coords, r=1000)
tier2_idx_2km = tier2_tree.query_ball_point(hdb_coords, r=2000)

unique_hdb["tier2_schools_within_1km"] = [
    [tier2_names[i] for i in idx_list] for idx_list in tier2_idx_1km]

unique_hdb["tier2_schools_within_2km"] = [
    [tier2_names[i] for i in idx_list] for idx_list in tier2_idx_2km]

unique_hdb["num_tier2_schools_1km"] = unique_hdb["tier2_schools_within_1km"].apply(len)
unique_hdb["num_tier2_schools_2km"] = unique_hdb["tier2_schools_within_2km"].apply(len)

3.4 
- nearest_tier1_primary_school_dist_m

In [17]:
# nearest Tier 1 primary schools
tier1_schools = schools_gdf[schools_gdf["school_tier"] == "Tier 1"].copy()
tier1_coords = np.array([(geom.x, geom.y) for geom in tier1_schools.geometry])
tier1_names = tier1_schools["school_tier"].tolist()

tier1_tree = cKDTree(tier1_coords)
tier1_distances, tier1_indices = tier1_tree.query(hdb_coords, k=1)

unique_hdb["nearest_tier1_primary_school_dist_m"] = tier1_distances

# print results
print(unique_hdb.head())    

   month  year        town flat_type storey_range  floor_area_sqm  \
0      1  2019  ANG MO KIO    3 ROOM     01 TO 03            67.0   
1      1  2019  ANG MO KIO    3 ROOM     01 TO 03            60.0   
2      1  2019  ANG MO KIO    3 ROOM     04 TO 06            67.0   
4      1  2019  ANG MO KIO    3 ROOM     01 TO 03            68.0   
5      1  2019  ANG MO KIO    3 ROOM     01 TO 03            67.0   

       flat_model     remaining_lease                     geometry  \
0  New Generation            58 years   POINT (28537.68 38825.233)   
1        Improved            66 years  POINT (28487.136 39687.765)   
2  New Generation            59 years    POINT (30336.26 38718.18)   
4  New Generation  60 years 08 months  POINT (29003.603 40255.313)   
5  New Generation  59 years 04 months  POINT (30406.267 39131.468)   

  postal_code  ...  num_schools_2km  \
0      560225  ...                9   
1      560174  ...                7   
2      560440  ...                9   
4      5

In [19]:
# merge back to full resale dataset if needed
school_features = unique_hdb[[
    "postal_code",
    "num_schools_1km",
    "num_schools_2km",
    "num_tier1_schools_1km",
    "num_tier1_schools_2km",
    "num_tier2_schools_1km",
    "num_tier2_schools_2km",
    "nearest_tier1_primary_school_dist_m"
]].copy()

resale_with_school = resale_2025_gdf.merge(
    school_features,
    on="postal_code",
    how="left"
)
resale_with_school.head(5)

,month,year,town,flat_type,storey_range,floor_area_sqm,flat_model,remaining_lease,geometry,postal_code,resale_price,num_schools_1km,num_schools_2km,num_tier1_schools_1km,num_tier1_schools_2km,num_tier2_schools_1km,num_tier2_schools_2km,nearest_tier1_primary_school_dist_m
0,1,2019,ANG MO KIO,3 ROOM,01 TO 03,67.0,New Generation,58 years,POINT (28537.68 38825.233),560225,356061.07,3,9,2,3,0,1,800.095964
1,1,2019,ANG MO KIO,3 ROOM,01 TO 03,60.0,Improved,66 years,POINT (28487.136 39687.765),560174,363801.53,3,7,1,2,0,1,427.710939
2,1,2019,ANG MO KIO,3 ROOM,04 TO 06,67.0,New Generation,59 years,POINT (30336.26 38718.18),560440,368445.80,3,9,0,1,0,1,1742.344901
3,1,2019,ANG MO KIO,3 ROOM,10 TO 12,61.0,Improved,66 years 01 month,POINT (28487.136 39687.765),560174,371541.98,3,7,1,2,0,1,427.710939
4,1,2019,ANG MO KIO,3 ROOM,01 TO 03,68.0,New Generation,60 years 08 months,POINT (29003.603 40255.313),560637,371541.98,2,6,0,1,1,1,1176.164205


### 4. Check data integrity 

In [20]:
# define logical conditions to check for data integrity
mask = (
    (resale_with_school['num_tier1_schools_1km'] > resale_with_school['num_schools_1km']) |
    (resale_with_school['num_tier1_schools_2km'] > resale_with_school['num_schools_2km']) |
    (resale_with_school['num_tier2_schools_1km'] > resale_with_school['num_schools_1km']) |
    (resale_with_school['num_tier2_schools_2km'] > resale_with_school['num_schools_2km'])
)

# check if any row returned True for the errors
if mask.any():
    error_count = mask.sum()
    raise ValueError(f"Data Integrity Error: {error_count} records have tier counts exceeding total school counts.")
else:
    print("Validation passed.")

Validation passed.


### 6. Merge with other features

In [21]:
# load data
mrt_df = pd.read_csv("../cleaned_datasets/hdb_mrt_distances.csv")
busstop_df = pd.read_csv("../cleaned_datasets/hdb_busstop_distances.csv")
hawker_df = pd.read_csv("../cleaned_datasets/hdb_hawker_distances.csv")
mall_df = pd.read_csv("../cleaned_datasets/hdb_mall_distances.csv")
cbd_df = pd.read_csv("../cleaned_datasets/hdb_cbd_distances.csv")

In [22]:
# standardize postal codes to 6-digit strings for merging
def standardize_postal_code(df, col="postal_code"):
    df = df.copy()
    df[col] = (
        df[col]
        .astype(str)
        .str.extract(r"(\d+)", expand=False)
        .str.zfill(6)
    )
    return df

In [32]:
# left join selected columns from feature_df into base_df using postal code.
# raises an error if feature_df has duplicate postal codes.
def merge_features(base_df, feature_df, feature_name, cols_to_add, rename_map=None,
                   base_postal_col="postal_code", feature_postal_col="postal_code"):
    
    rename_map = rename_map or {}

    base_df = standardize_postal_code(base_df, base_postal_col)
    feature_df = standardize_postal_code(feature_df, feature_postal_col)

    # keep only what is needed
    use_cols = [feature_postal_col] + cols_to_add
    missing_cols = [c for c in use_cols if c not in feature_df.columns]
    if missing_cols:
        raise KeyError(f"{feature_name}: missing columns in feature_df -> {missing_cols}")

    temp = feature_df[use_cols].copy()

    # rename feature columns if needed
    temp = temp.rename(columns=rename_map)

    # make sure no duplicate postal codes in feature table
    dupes = temp[feature_postal_col].duplicated().sum()
    if dupes > 0:
        duplicate_rows = temp[temp[feature_postal_col].duplicated(keep=False)].sort_values(feature_postal_col)
        raise ValueError(
            f"{feature_name}: found {dupes} duplicate postal codes. "
            f"Deduplicate before merging.\nSample duplicates:\n{duplicate_rows.head()}"
        )

    merged = base_df.merge(
        temp,
        left_on=base_postal_col,
        right_on=feature_postal_col,
        how="left"
    )

    # drop duplicated postal column from right side if different name used
    if feature_postal_col != base_postal_col and feature_postal_col in merged.columns:
        merged = merged.drop(columns=[feature_postal_col])

    return merged

In [33]:
resale_features_df = resale_with_school.copy()

# 1) MRT features
# Match by postal code
# Add:
#   num_unique_mrt_500m
#   num_unique_mrt_1km
#   num_unique_mrt_lines_500m
#   num_unique_mrt_lines_1km
#   walking_dist_mrt_m
resale_features_df = merge_features(
    base_df=resale_features_df,
    feature_df=mrt_df,
    feature_name="mrt_df",
    cols_to_add=[
        "num_unique_mrt_500m",
        "num_unique_mrt_1km",
        "num_unique_mrt_lines_500m",
        "num_unique_mrt_lines_1km",
        "walking_dist_mrt_m",
    ],
    base_postal_col="postal_code",
    feature_postal_col="postal_code"
)

# 2) Bus stop features
# Match by postal code
# Add:
#   walking_dist_m  -> rename to walking_dist_busstop_m
#   num_busstops_500m
#   num_busstops_1km
resale_features_df = merge_features(
    base_df=resale_features_df,
    feature_df=busstop_df,
    feature_name="busstop_df",
    cols_to_add=[
        "num_busstops_500m",
        "num_busstops_1km",
        "walking_dist_m",
    ],
    rename_map={
        "walking_dist_m": "walking_dist_busstop_m"
    },
    base_postal_col="postal_code",
    feature_postal_col="postal_code"
)

# 3) Hawker features
# Match by postal code
# Add:
#   num_unique_hawker_500m
#   num_unique_hawker_1km
#   walking_dist_hawker_m
resale_features_df = merge_features(
    base_df=resale_features_df,
    feature_df=hawker_df,
    feature_name="hawker_df",
    cols_to_add=[
        "num_unique_hawker_500m",
        "num_unique_hawker_1km",
        "walking_dist_hawker_m",
    ],
    base_postal_col="postal_code",
    feature_postal_col="postal_code"
)

# 4) Mall features
# Match by postal code
# Add:
#   walking_dist_m -> rename to walking_dist_mall_m
#   num_malls_1km
#   num_malls_2km
resale_features_df = merge_features(
    base_df=resale_features_df,
    feature_df=mall_df,
    feature_name="mall_df",
    cols_to_add=[
        "num_malls_1km",
        "num_malls_2km",
        "walking_dist_m",
    ],
    rename_map={
        "walking_dist_m": "walking_dist_mall_m"
    },
    base_postal_col="postal_code",
    feature_postal_col="postal_code"
)

# 5) CBD features
# Match by postal code
# Add:
#   dist_to_cdb_m -> rename to dist_cbd_m
resale_features_df = merge_features(
    base_df=resale_features_df,
    feature_df=cbd_df,
    feature_name="cbd_df",
    cols_to_add=[
        "dist_to_cbd_m",
    ],
    rename_map={
        "dist_to_cbd_m": "dist_cbd_m"
    },
    base_postal_col="postal_code",
    feature_postal_col="postal_code"
)

In [34]:
missing_values = resale_features_df.isna().sum()
if missing_values.any():
    print(missing_values[missing_values > 0])
else:
    print("No missing values in joined feature columns.")
# print
resale_features_df.head()
resale_features_df.info()

No missing values in joined feature columns.
<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 180003 entries, 0 to 180002
Data columns (total 33 columns):
 #   Column                               Non-Null Count   Dtype   
---  ------                               --------------   -----   
 0   month                                180003 non-null  int32   
 1   year                                 180003 non-null  int32   
 2   town                                 180003 non-null  str     
 3   flat_type                            180003 non-null  str     
 4   storey_range                         180003 non-null  str     
 5   floor_area_sqm                       180003 non-null  float64 
 6   flat_model                           180003 non-null  str     
 7   remaining_lease                      180003 non-null  str     
 8   geometry                             180003 non-null  geometry
 9   postal_code                          180003 non-null  str     
 10  resale_price   

In [26]:
resale_features_df.to_csv("../datasets/resale_with_all_features.csv", index=False)
resale_features_df.to_parquet('../cleaned_datasets/resale_with_all_features.parquet', index=False)

### 7. Dummy encode

In [27]:
categorical_cols = ["town", "flat_type", "storey_range", "flat_model"]

resale_features_dummies = pd.get_dummies(
    resale_features_df,
    columns=categorical_cols,
    drop_first=True,  
    dtype=int
)

print(resale_features_dummies.shape)
print(resale_features_dummies.columns.tolist())

(180003, 96)
['month', 'year', 'floor_area_sqm', 'remaining_lease', 'geometry', 'postal_code', 'resale_price', 'num_schools_1km', 'num_schools_2km', 'num_tier1_schools_1km', 'num_tier1_schools_2km', 'num_tier2_schools_1km', 'num_tier2_schools_2km', 'nearest_tier1_primary_school_dist_m', 'num_unique_mrt_500m', 'num_unique_mrt_1km', 'num_unique_mrt_lines_500m', 'num_unique_mrt_lines_1km', 'walking_dist_mrt_m', 'num_busstops_500m', 'num_busstops_1km', 'walking_dist_busstop_m', 'num_unique_hawker_500m', 'num_unique_hawker_1km', 'walking_dist_hawker_m', 'num_malls_1km', 'num_malls_2km', 'walking_dist_mall_m', 'dist_cbd_m', 'town_BEDOK', 'town_BISHAN', 'town_BUKIT BATOK', 'town_BUKIT MERAH', 'town_BUKIT PANJANG', 'town_BUKIT TIMAH', 'town_CENTRAL AREA', 'town_CHOA CHU KANG', 'town_CLEMENTI', 'town_GEYLANG', 'town_HOUGANG', 'town_JURONG EAST', 'town_JURONG WEST', 'town_KALLANG/WHAMPOA', 'town_MARINE PARADE', 'town_PASIR RIS', 'town_PUNGGOL', 'town_QUEENSTOWN', 'town_SEMBAWANG', 'town_SENGKANG

In [28]:
resale_features_dummies.to_csv("../datasets/resale_with_all_features_dummies.csv", index=False)
resale_features_dummies.to_parquet('../cleaned_datasets/resale_with_all_features_dummies.parquet', index=False)